# 04 — Locked external validation and robustness

This notebook is intentionally a **PENDING / read-only** gate until the full H1-N comparison is complete and its frozen evaluation plan is recorded. It does not use external data for training, early stopping, threshold selection, representation selection, or robustness tuning.

In [ ]:
from pathlib import Path
import json

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL

ARTIFACT_ROOT = Path('../artifacts')

def completed_h1n_runs() -> list[dict[str, object]]:
    rows = []
    for run_path in ARTIFACT_ROOT.glob('*/run.json'):
        metrics_path = run_path.parent / 'internal_test_metrics.json'
        if not metrics_path.is_file():
            continue
        run = json.loads(run_path.read_text(encoding='utf-8'))
        if run.get('preprocessing', {}).get('protocol') != CONTROLLED_PREPROCESSING_PROTOCOL:
            continue
        config = run.get('config', {})
        rows.append({
            'run': run_path.parent.name,
            'representation': config.get('representation'),
            'seed': config.get('seed'),
            'threshold': run.get('threshold'),
            'metrics_path': str(metrics_path),
        })
    return sorted(rows, key=lambda row: (str(row['representation']), int(row['seed'])))

internal_runs = completed_h1n_runs()
if internal_runs:
    display(pd.DataFrame(internal_runs))
else:
    print('PENDING: no completed controlled H1-N internal result is available.')

## External locked-test plan

1. Complete the six predeclared H1-N internal runs and their cluster-aware analysis. Do not choose a winning seed.
2. Freeze the representation comparison, three-seed aggregation, checkpoint selection rule, validation-selected threshold rule, and common-raster preprocessing.
3. Obtain Synthbuster and RAISE-1k under their research licences, record the local archive hashes, and create a separate external-only manifest. The preparation command never downloads data.
4. Audit exact-file and perceptual-hash candidate overlap against Defactify before inference. Any overlap handling is documented before metrics are calculated.
5. Run the frozen model exactly once on the external manifest, store every prediction, and report generator-specific, macro and worst-generator values without changing the model or threshold.

Because the Defactify test rows were already read during D0, H1-N metrics there are exploratory internal stress tests. The external corpus is the confirmatory evaluation; a good internal score cannot replace it.

In [ ]:
import shlex
import sys

def external_manifest_command(
    synthetic_root: Path,
    raise_root: Path,
    output_root: Path,
) -> list[str]:
    """Build the manifest only after the H1-N plan is frozen and local data are licensed."""
    return [
        sys.executable,
        'scripts/prepare_synthbuster_external.py',
        '--synthetic-root', str(synthetic_root),
        '--raise-root', str(raise_root),
        '--output-root', str(output_root),
        '--reference-manifest', 'data/processed/defactify_grouped/manifest.csv',
    ]

print('External manifest preparation is locked: configure real, licensed local paths only after freezing H1-N.')
print('The function above returns a valid command once its three Path arguments are supplied.')

## Robustness status

The evaluator now restores the selected run's preprocessing through `ModelBundle` and applies every fixed JPEG/resize/blur condition **after** the H1-N common raster. The frozen checkpoint and validation-selected threshold must be reused; the clean internal predictions must never be overwritten. The gated command below is H1-N-valid only after one experiment has been explicitly frozen. Its Defactify result remains an exploratory internal stress test because D0 already opened that test split.

In [ ]:
import os
import shlex
import subprocess
import sys

def robustness_command(frozen_experiment: str) -> list[str]:
    """Build the H1-N robustness command for one explicitly frozen experiment."""
    if Path(frozen_experiment).name != frozen_experiment:
        raise ValueError('Use one experiment directory name, not a path.')
    experiment_dir = Path('artifacts') / frozen_experiment
    return [
        sys.executable,
        'scripts/evaluate_robustness.py',
        '--manifest', 'data/processed/defactify_grouped/manifest.csv',
        '--experiment-dir', str(experiment_dir),
        '--output-dir', str(experiment_dir / 'robustness'),
    ]

FROZEN_EXPERIMENT = os.environ.get('H1N_FROZEN_EXPERIMENT')
if FROZEN_EXPERIMENT:
    ROBUSTNESS_COMMAND = robustness_command(FROZEN_EXPERIMENT)
    print(shlex.join(ROBUSTNESS_COMMAND))
    if os.environ.get('RUN_H1N_ROBUSTNESS') == '1':
        subprocess.run(ROBUSTNESS_COMMAND, check=True, cwd=Path('..').resolve())
    else:
        print('PENDING: command is printed only; set RUN_H1N_ROBUSTNESS=1 after the freeze to run it.')
else:
    print('PENDING: set H1N_FROZEN_EXPERIMENT only after documenting the frozen checkpoint and threshold.')

In [ ]:
robustness_tables = []
for run in internal_runs:
    metrics_path = ARTIFACT_ROOT / str(run['run']) / 'robustness' / 'robustness_metrics.csv'
    if metrics_path.is_file():
        table = pd.read_csv(metrics_path)
        table.insert(0, 'run', run['run'])
        robustness_tables.append(table)

if robustness_tables:
    display(pd.concat(robustness_tables, ignore_index=True))
else:
    print('PENDING: no H1-N-verified robustness artifact is available.')

## Permitted conclusion

A high clean score with poor external or JPEG performance is evidence of distribution-specific artefacts, not a deployment-ready detector. A model score remains a score, not a calibrated probability or proof of origin. The web interface, if ever enabled, must expose this limitation and use only a model frozen after the external evaluation.